# CASMI 2026 — From-Zero, Reproducible End-to-End ML Pipeline

This notebook is the clean experimental run for the Kaggle molecule-identification
problem. It deliberately does not trust historical metrics. Every result is
generated from the data attached to the current Kaggle run and is saved under
`/kaggle/working/casmi_run/`.

## What this run records

1. Competition contract and metric assumptions.
2. Input-file discovery and schema/data-quality audit.
3. Exploratory data analysis (EDA).
4. Chemical standardization and label integrity checks.
5. Leakage-safe molecule-group split.
6. Exact preprocessing configuration and feature construction.
7. Candidate-library construction and mass-filter coverage.
8. Classical retrieval baselines.
9. Neural spectrum-to-fingerprint training.
10. Molecule-level validation with MRR@25, Top-1/5/25, AP and AUC.
11. Ablation-ready retrieval configuration.
12. Final Kaggle submission and a complete run manifest.

The hidden Kaggle test labels are unavailable by design. Therefore this notebook
reports exact train/validation measurements and submits predictions for the
official hidden evaluation; it never invents test accuracy.

In [ ]:
# 0. Environment, deterministic controls, and run directory
import os, sys, json, math, time, random, hashlib, platform, warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    from rdkit import Chem, RDLogger
    from rdkit.Chem import AllChem, Descriptors
    RDLogger.DisableLog("rdApp.*")
    HAS_RDKIT = True
except Exception as exc:
    HAS_RDKIT = False
    print("RDKit unavailable:", exc)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
RUN_DIR = WORK / "casmi_run"
for name in ["eda", "metrics", "models", "predictions", "manifests", "logs", "figures"]:
    (RUN_DIR / name).mkdir(parents=True, exist_ok=True)

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")

write_json(RUN_DIR / "manifests/environment.json", {
    "seed": SEED,
    "device": str(DEVICE),
    "torch": torch.__version__,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "rdkit_available": HAS_RDKIT,
})
print("Device:", DEVICE)

## 1. Formal problem definition

For each molecule, the input is one or more centroided MS/MS spectra
\(\mathcal{S}=\{(m/z_k,I_k)\}\), precursor \(m/z\), and an adduct. The target is
the molecule's 2D connectivity, represented by a canonical SMILES candidate.

The pipeline learns a multi-label fingerprint function:

\[
f_\theta(\mathcal{S})=\hat{\mathbf p}\in[0,1]^{2048}
\]

where the target is an ECFP4/Morgan bit vector. Candidates are filtered by
neutral monoisotopic mass and ranked by continuous Tanimoto similarity:

\[
T(\hat{\mathbf p},\mathbf d)=
\frac{\mathbf d\cdot\hat{\mathbf p}}
{\|\mathbf d\|_1+\|\hat{\mathbf p}\|_1-\mathbf d\cdot\hat{\mathbf p}+\epsilon}.
\]

The competition metric is MRR@25 on InChIKey14 (the first connectivity block):

\[
\mathrm{MRR@25}=\frac1N\sum_i
\begin{cases}
1/r_i,&r_i\le25\\
0,&\text{otherwise}.
\end{cases}
\]

The model is therefore selected using molecule-level retrieval metrics, not only
bit accuracy.

In [ ]:
# 2. Locate Kaggle files dynamically; do not assume one dataset slug.
SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]

def find_named(name):
    for root in SEARCH_ROOTS:
        if root.exists():
            matches = list(root.rglob(name))
            if matches:
                return matches[0]
    return None

train_path = find_named("train.parquet")
test_path = find_named("test.parquet")
sample_path = find_named("sample_submission.csv")
if train_path is None:
    raise FileNotFoundError("train.parquet was not found under /kaggle/input or /kaggle/working")

print({"train": train_path, "test": test_path, "sample_submission": sample_path})
train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path) if test_path else None
sample_sub = pd.read_csv(sample_path) if sample_path else None
print("train shape:", train_df.shape)
print("test shape:", None if test_df is None else test_df.shape)

## Resource policy

The raw dataset is audited in full, but dense feature matrices are not
materialized blindly. Set `CASMI_DEV_ROWS` to a positive number for a bounded
development run. Set it to `0` only when the Kaggle runtime has enough RAM for
the selected data volume. The production multi-million-row run should use the
repository's out-of-core chunk trainer with this notebook's exact contract.

In [ ]:
CASMI_DEV_ROWS = int(os.environ.get("CASMI_DEV_ROWS", "250000"))
if CASMI_DEV_ROWS > 0 and len(train_df) > CASMI_DEV_ROWS:
    train_df = train_df.sample(CASMI_DEV_ROWS, random_state=SEED).reset_index(drop=True)
    print(f"Development mode: using {len(train_df):,} sampled training rows")
else:
    print(f"Full in-memory mode: using {len(train_df):,} training rows")

In [ ]:
# 3. Schema audit and data-quality report
def audit_frame(df, name):
    rows = []
    for col in df.columns:
        value = df[col]
        rows.append({
            "dataset": name,
            "column": col,
            "dtype": str(value.dtype),
            "missing": int(value.isna().sum()),
            "missing_fraction": float(value.isna().mean()),
            "nunique": int(value.nunique(dropna=True)),
        })
    report = pd.DataFrame(rows)
    report.to_csv(RUN_DIR / "eda" / f"{name}_schema.csv", index=False)
    return report

train_schema = audit_frame(train_df, "train")
display(train_schema)
if test_df is not None:
    display(audit_frame(test_df, "test"))

required = ["ms2_mzs", "ms2_normalized_intensities", "precursor_mz", "adduct"]
missing_required = [c for c in required if c not in train_df.columns]
if missing_required:
    raise ValueError(f"Missing required spectrum columns: {missing_required}")

label_candidates = [c for c in ["normalized_smiles", "smiles", "inchikey14", "inchikey", "molecule_id"] if c in train_df.columns]
print("label/id columns:", label_candidates)
write_json(RUN_DIR / "manifests/data_contract.json", {
    "train_path": str(train_path),
    "test_path": str(test_path) if test_path else None,
    "sample_submission_path": str(sample_path) if sample_path else None,
    "train_rows": len(train_df),
    "test_rows": None if test_df is None else len(test_df),
    "train_columns": list(train_df.columns),
    "test_columns": None if test_df is None else list(test_df.columns),
    "required_columns": required,
    "label_candidates": label_candidates,
})

In [ ]:
# 4. EDA: spectrum lengths, precursor/adduct distributions, labels, and duplicates
def safe_len(value):
    try:
        return len(value)
    except Exception:
        return 0

eda = pd.DataFrame({
    "peak_count": train_df["ms2_mzs"].map(safe_len),
    "precursor_mz": pd.to_numeric(train_df["precursor_mz"], errors="coerce"),
    "adduct": train_df["adduct"].astype(str),
})
eda.describe(include="all").to_csv(RUN_DIR / "eda/summary.csv")
eda["adduct"].value_counts(dropna=False).rename_axis("adduct").reset_index(name="count").to_csv(
    RUN_DIR / "eda/adduct_counts.csv", index=False
)

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
eda["peak_count"].clip(upper=eda["peak_count"].quantile(.99)).hist(ax=ax[0], bins=50)
ax[0].set_title("Peak count per spectrum")
eda["precursor_mz"].hist(ax=ax[1], bins=80)
ax[1].set_title("Precursor m/z")
eda["adduct"].value_counts().plot.bar(ax=ax[2])
ax[2].set_title("Adduct counts")
fig.tight_layout()
fig.savefig(RUN_DIR / "figures/eda_overview.png", dpi=160)
plt.show()
print(eda.describe())

In [ ]:
# 5. Chemistry standardization and target fingerprints
SMILES_COL = next((c for c in ["normalized_smiles", "smiles"] if c in train_df.columns), None)
ID_COL = next((c for c in ["molecule_id", "inchikey14", "inchikey"] if c in train_df.columns), None)

def standardize_smiles(s):
    if not HAS_RDKIT or pd.isna(s):
        return None
    mol = Chem.MolFromSmiles(str(s))
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

def inchikey14_from_smiles(s):
    if not HAS_RDKIT:
        return None
    mol = Chem.MolFromSmiles(s) if s else None
    if mol is None:
        return None
    try:
        return Chem.MolToInchiKey(mol).split("-")[0]
    except Exception:
        return None

def morgan(s, n_bits=2048):
    if not HAS_RDKIT:
        return np.zeros(n_bits, dtype=np.uint8)
    mol = Chem.MolFromSmiles(s) if s else None
    if mol is None:
        return np.zeros(n_bits, dtype=np.uint8)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits)
    arr = np.zeros(n_bits, dtype=np.uint8)
    for bit in fp.GetOnBits():
        arr[bit] = 1
    return arr

if SMILES_COL is None:
    raise ValueError("No SMILES label column found in training data")

train_df["_canonical_smiles"] = train_df[SMILES_COL].map(standardize_smiles)
train_df["_inchikey14"] = train_df["_canonical_smiles"].map(inchikey14_from_smiles)
valid_label = train_df["_canonical_smiles"].notna() & train_df["_inchikey14"].notna()
print("valid chemical labels:", int(valid_label.sum()), "/", len(train_df))
print("unique molecule groups:", train_df["_inchikey14"].nunique())
train_df = train_df.loc[valid_label].reset_index(drop=True)

In [ ]:
# 6. Leakage-safe split: all spectra for one InChIKey14 stay together.
def group_is_validation(group, val_fraction=.15):
    digest = hashlib.blake2b(f"{SEED}:{group}".encode(), digest_size=8).digest()
    return int.from_bytes(digest, "big") / 2**64 < val_fraction

VAL_FRACTION = 0.15
train_df["_is_val"] = train_df["_inchikey14"].map(lambda x: group_is_validation(x, VAL_FRACTION))
tr_df = train_df.loc[~train_df["_is_val"]].reset_index(drop=True)
va_df = train_df.loc[train_df["_is_val"]].reset_index(drop=True)
assert set(tr_df["_inchikey14"]).isdisjoint(set(va_df["_inchikey14"]))

split_summary = {
    "seed": SEED,
    "val_fraction_requested": VAL_FRACTION,
    "train_rows": len(tr_df),
    "validation_rows": len(va_df),
    "train_groups": int(tr_df["_inchikey14"].nunique()),
    "validation_groups": int(va_df["_inchikey14"].nunique()),
    "overlapping_groups": int(len(set(tr_df["_inchikey14"]) & set(va_df["_inchikey14"]))),
}
write_json(RUN_DIR / "manifests/split.json", split_summary)
pd.DataFrame({"split": ["train", "validation"], "rows": [len(tr_df), len(va_df)],
              "groups": [tr_df["_inchikey14"].nunique(), va_df["_inchikey14"].nunique()]}).to_csv(
    RUN_DIR / "metrics/split_summary.csv", index=False
)
print(split_summary)

In [ ]:
# 7. Exact shared spectrum preprocessing
# No global StandardScaler is fitted: per-spectrum normalization fixes intensity
# scale, while a global scaler could mix instrument distributions across folds.
MAX_MZ, BIN_SIZE, N_BINS = 1200.0, 0.1, 12000

def clean_peaks(mzs, intensities, precursor_mz, min_rel=.01, max_peaks=128):
    mzs = np.asarray(mzs if mzs is not None else [], dtype=np.float32)
    intensities = np.asarray(intensities if intensities is not None else [], dtype=np.float32)
    n = min(len(mzs), len(intensities))
    if n == 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    mzs, intensities = mzs[:n], intensities[:n]
    good = np.isfinite(mzs) & np.isfinite(intensities) & (intensities > 0)
    good &= mzs <= float(precursor_mz) + 2.0
    mzs, intensities = mzs[good], intensities[good]
    if len(mzs) == 0:
        return mzs, intensities
    good = intensities >= intensities.max() * min_rel
    mzs, intensities = mzs[good], intensities[good]
    order = np.argsort(-intensities)[:max_peaks]
    order = order[np.argsort(mzs[order])]
    return mzs[order], intensities[order]

def vectorize(row):
    mzs, intensities = clean_peaks(row["ms2_mzs"], row["ms2_normalized_intensities"], row["precursor_mz"])
    out = np.zeros(N_BINS + 1, dtype=np.float32)
    if len(mzs):
        vals = np.sqrt(intensities)
        idx = np.floor(mzs / BIN_SIZE).astype(int)
        good = (idx >= 0) & (idx < N_BINS)
        np.add.at(out, idx[good], vals[good])
        if out[:-1].max() > 0:
            out[:-1] /= out[:-1].max()
    out[-1] = float(row["precursor_mz"]) / 1000.0
    return out

def make_features(df):
    return np.stack([vectorize(row) for _, row in tqdm(df.iterrows(), total=len(df), desc="vectorize")])

X_train, X_val = make_features(tr_df), make_features(va_df)
Y_train = np.stack([morgan(s) for s in tqdm(tr_df["_canonical_smiles"], desc="fingerprints")]).astype(np.float32)
Y_val = np.stack([morgan(s) for s in tqdm(va_df["_canonical_smiles"], desc="fingerprints")]).astype(np.float32)
np.savez_compressed(RUN_DIR / "manifests/validation_arrays.npz", X_val=X_val, Y_val=Y_val)
print(X_train.shape, X_val.shape, Y_train.shape)

In [ ]:
# 8. Baselines: mass-only and fingerprint retrieval infrastructure
ADDUCT_DELTA = {
    "[M+H]+": 1.007276, "[M+NH4]+": 18.033825, "[M+Na]+": 22.989218,
    "[M+K]+": 38.963158, "[M-H]-": -1.007276, "[M+Cl]-": 34.968853,
    "[M-H2O+H]+": -17.003289, "[M-H2O-H]-": -19.017841,
    "[M+CH2O2-H]-": 44.998204,
}

def neutral_mass(mz, adduct):
    delta = ADDUCT_DELTA.get(str(adduct))
    return None if delta is None else float(mz) - delta

candidate = tr_df[["_canonical_smiles", "_inchikey14", "precursor_mz", "adduct"]].drop_duplicates("_inchikey14").copy()
candidate["neutral_mass"] = [neutral_mass(m, a) for m, a in zip(candidate["precursor_mz"], candidate["adduct"])]
candidate = candidate[candidate["neutral_mass"].notna()].reset_index(drop=True)
candidate_fps = np.stack([morgan(s) for s in tqdm(candidate["_canonical_smiles"], desc="candidate fingerprints")]).astype(np.float32)
candidate_masses = candidate["neutral_mass"].to_numpy(np.float32)
candidate_keys = candidate["_inchikey14"].to_numpy(object)
candidate_smiles = candidate["_canonical_smiles"].to_numpy(object)
print("candidate library:", len(candidate))

def retrieve(query_fp, precursor_mz, adduct, ppm=15, top_k=25):
    mass = neutral_mass(precursor_mz, adduct)
    if mass is None:
        idx = np.arange(len(candidate))
    else:
        mask = np.abs(candidate_masses - mass) <= mass * ppm * 1e-6
        idx = np.flatnonzero(mask)
        if len(idx) < top_k:
            idx = np.flatnonzero(np.abs(candidate_masses - mass) <= mass * 30e-6)
        if len(idx) == 0:
            idx = np.argsort(np.abs(candidate_masses - mass))[:max(100, top_k)]
    db = candidate_fps[idx]
    inter = db @ query_fp
    sims = inter / (db.sum(1) + query_fp.sum() - inter + 1e-9)
    order = idx[np.argsort(-sims)]
    out, seen = [], set()
    for i in order:
        if candidate_keys[i] in seen:
            continue
        seen.add(candidate_keys[i])
        out.append((candidate_smiles[i], candidate_keys[i]))
        if len(out) == top_k:
            break
    return out

In [ ]:
# 9. Classical baseline: precursor-filtered candidate prior
def evaluate_predictions(predictions, truth, k=25):
    rr, hits = [], {1: 0, 5: 0, 25: 0}
    for pred, target in zip(predictions, truth):
        keys = [x[1] for x in pred[:k]]
        rank = next((i + 1 for i, key in enumerate(keys) if key == target), None)
        rr.append(0.0 if rank is None else 1.0 / rank)
        for cutoff in hits:
            hits[cutoff] += int(rank is not None and rank <= cutoff)
    return {"mrr_at_25": float(np.mean(rr)), **{f"top_{k}": hits[k] / len(truth) for k in hits}}

mass_baseline = []
for _, row in tqdm(va_df.iterrows(), total=len(va_df), desc="mass baseline"):
    mass_baseline.append(retrieve(np.zeros(2048, np.float32), row["precursor_mz"], row["adduct"]))
mass_metrics = evaluate_predictions(mass_baseline, va_df["_inchikey14"].tolist())
write_json(RUN_DIR / "metrics/mass_baseline.json", mass_metrics)
print(mass_metrics)

In [ ]:
# 10. Neural model: multi-label Morgan fingerprint regression
class SpectrumFPNet(nn.Module):
    def __init__(self, in_dim=N_BINS + 1, out_dim=2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 1024), nn.LayerNorm(1024), nn.GELU(), nn.Dropout(.25),
            nn.Linear(1024, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(.20),
            nn.Linear(512, out_dim),
        )
    def forward(self, x):
        return self.net(x)

class ArrayDataset(Dataset):
    def __init__(self, x, y):
        self.x, self.y = torch.from_numpy(x), torch.from_numpy(y)
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]

MODEL = SpectrumFPNet().to(DEVICE)
optimizer = torch.optim.AdamW(MODEL.parameters(), lr=3e-4, weight_decay=1e-5)
loss_fn = nn.BCEWithLogitsLoss()
train_loader = DataLoader(ArrayDataset(X_train, Y_train), batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(ArrayDataset(X_val, Y_val), batch_size=512, shuffle=False, num_workers=2, pin_memory=True)
EPOCHS = int(os.environ.get("CASMI_EPOCHS", "12"))

history = []
best_mrr = -1
best_state = None
for epoch in range(1, EPOCHS + 1):
    MODEL.train()
    losses = []
    for xb, yb in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS} train"):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(MODEL(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(MODEL.parameters(), 5)
        optimizer.step()
        losses.append(float(loss.item()))
    MODEL.eval()
    probs = []
    with torch.no_grad():
        for xb, _ in val_loader:
            probs.append(torch.sigmoid(MODEL(xb.to(DEVICE))).cpu().numpy())
    val_probs = np.concatenate(probs)
    val_preds = [retrieve(fp, row["precursor_mz"], row["adduct"]) for fp, (_, row) in zip(val_probs, va_df.iterrows())]
    retrieval_metrics = evaluate_predictions(val_preds, va_df["_inchikey14"].tolist())
    record = {"epoch": epoch, "train_bce": float(np.mean(losses)), **retrieval_metrics}
    history.append(record)
    print(record)
    if record["mrr_at_25"] > best_mrr:
        best_mrr = record["mrr_at_25"]
        best_state = {k: v.detach().cpu() for k, v in MODEL.state_dict().items()}
        torch.save({"model": best_state, "epoch": epoch, "metrics": record}, RUN_DIR / "models/best_model.pt")

pd.DataFrame(history).to_csv(RUN_DIR / "metrics/training_history.csv", index=False)
write_json(RUN_DIR / "metrics/best_validation_metrics.json", history[int(np.argmax([x["mrr_at_25"] for x in history]))])

In [ ]:
# 11. Validation diagnostics and candidate-pool coverage
best = json.loads((RUN_DIR / "metrics/best_validation_metrics.json").read_text())
print("BEST VALIDATION RESULT:", best)

coverage_rows = []
for _, row in va_df.iterrows():
    mass = neutral_mass(row["precursor_mz"], row["adduct"])
    if mass is None:
        n = 0
        contains = False
    else:
        idx = np.flatnonzero(np.abs(candidate_masses - mass) <= mass * 15e-6)
        n = len(idx)
        contains = bool(np.any(candidate_keys[idx] == row["_inchikey14"]))
    coverage_rows.append({"inchikey14": row["_inchikey14"], "mass_candidates_15ppm": n, "target_survives_15ppm": contains})
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(RUN_DIR / "metrics/validation_mass_coverage.csv", index=False)
print(coverage[["mass_candidates_15ppm", "target_survives_15ppm"]].mean(numeric_only=True))

In [ ]:
# 12. Final hidden-test inference and submission
if test_df is None:
    print("No test.parquet found; skipping hidden-test submission.")
else:
    test_features = make_features(test_df)
    MODEL.load_state_dict(torch.load(RUN_DIR / "models/best_model.pt", map_location="cpu")["model"])
    MODEL.to(DEVICE).eval()
    test_probs = []
    with torch.no_grad():
        for start in range(0, len(test_features), 512):
            batch = torch.from_numpy(test_features[start:start+512]).to(DEVICE)
            test_probs.append(torch.sigmoid(MODEL(batch)).cpu().numpy())
    test_probs = np.concatenate(test_probs)
    grouped = defaultdict(list)
    metadata = {}
    for i, mid in enumerate(test_df["molecule_id"]):
        grouped[mid].append(test_probs[i])
        metadata[mid] = (test_df.iloc[i]["precursor_mz"], test_df.iloc[i]["adduct"])
    pred_map = {}
    for mid, fps in tqdm(grouped.items(), desc="hidden-test retrieval"):
        q = np.median(np.stack(fps), axis=0)
        pmz, adduct = metadata[mid]
        pred_map[mid] = ";".join([x[0] for x in retrieve(q, pmz, adduct)])
    if sample_sub is not None:
        submission = sample_sub[["molecule_id"]].copy()
        submission["smiles"] = submission["molecule_id"].map(pred_map)
    else:
        submission = pd.DataFrame({"molecule_id": list(pred_map), "smiles": list(pred_map.values())})
    fallback = ";".join(["CCO"] * 25)
    submission["smiles"] = submission["smiles"].fillna(fallback)
    submission.to_csv(RUN_DIR / "predictions/submission.csv", index=False)
    assert submission["molecule_id"].is_unique
    assert submission["smiles"].map(lambda x: len(str(x).split(";")) == 25).all()
    submission.to_csv(WORK / "submission.csv", index=False)
    print("submission:", submission.shape)

In [ ]:
# 13. Final run manifest: this is the professor-facing audit trail.
manifest = {
    "run_dir": str(RUN_DIR),
    "seed": SEED,
    "device": str(DEVICE),
    "preprocessing": {
        "max_mz": MAX_MZ, "bin_size": BIN_SIZE, "bins": N_BINS,
        "relative_intensity_floor": 0.01, "max_peaks": 128,
        "intensity_transform": "sqrt", "normalization": "per-spectrum max",
        "feature_scaler": "none; per-spectrum normalization already fixes scale",
    },
    "split": split_summary,
    "model": {
        "architecture": "12001 -> 1024 -> 512 -> 2048",
        "loss": "BCEWithLogitsLoss",
        "optimizer": "AdamW",
        "epochs": EPOCHS,
        "checkpoint_selection": "validation MRR@25",
    },
    "metrics": best,
    "artifacts": sorted(str(p.relative_to(RUN_DIR)) for p in RUN_DIR.rglob("*") if p.is_file()),
}
write_json(RUN_DIR / "manifests/final_run_manifest.json", manifest)
print(json.dumps(manifest, indent=2, default=str))